In [ ]:
import pandas as pd
import numpy as np
import re

csv_path = "/kaggle/input/datasets/amarnathdj/dec-2024-mechanical-rainfall/december_2024_mechanical_rainfall_data.csv"

# =====================================================
# LOAD
# =====================================================

df = pd.read_csv(csv_path)

print("Original shape:", df.shape)

# Remove index columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# =====================================================
# FIND COLUMNS
# =====================================================

time_col = None
rain_col = None

for col in df.columns:

    if "time" in col.lower():
        time_col = col

    if "rain" in col.lower():
        rain_col = col

print("Time column:", time_col)
print("Rain column:", rain_col)

# =====================================================
# CONVERT RAINFALL
# =====================================================

def convert_to_mm(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Extract first number
    match = re.search(r"[\d.]+", value)

    if match is None:
        return np.nan

    num = float(match.group())

    if "µm" in value:
        return num / 1000

    return num

# =====================================================
# STANDARDIZE
# =====================================================

df = df.rename(columns={
    time_col: "timestamp",
    rain_col: "rainfall_mm"
})

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="mixed",
    errors="coerce"
)

df["rainfall_mm"] = df["rainfall_mm"].apply(
    convert_to_mm
)

# =====================================================
# CHECK BEFORE DROPPING
# =====================================================

print("\nNaN timestamps:",
      df["timestamp"].isna().sum())

print("NaN rainfall:",
      df["rainfall_mm"].isna().sum())

# =====================================================
# CLEAN
# =====================================================

df = df.dropna(
    subset=["timestamp", "rainfall_mm"]
)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

# =====================================================
# SAVE
# =====================================================

output_path = "/kaggle/working/cleaned_labels_december_2024/December_2024_clean.csv"

df.to_csv(
    output_path,
    index=False
)

print("\nFinal shape:", df.shape)
print("\nSaved to:")
print(output_path)

print("\nHead:")
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re

# =====================================================
# CONFIG
# =====================================================

INPUT_FOLDER = "/kaggle/input/datasets/amarnathdj/oct-2025-mech-rainfall-data"

OUTPUT_FOLDER = "/kaggle/working/cleaned_labels_october_2025"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

# =====================================================
# FIND CSV FILES
# =====================================================

csv_files = glob.glob(
    os.path.join(INPUT_FOLDER, "**/*.csv"),
    recursive=True
)

print(f"Found {len(csv_files)} CSV files")

# =====================================================
# RAINFALL CONVERSION
# =====================================================

def convert_to_mm(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    match = re.search(
        r"[\d.]+",
        value
    )

    if match is None:
        return np.nan

    num = float(
        match.group()
    )

    if "µm" in value:
        return num / 1000

    return num

# =====================================================
# PROCESS EACH CSV
# =====================================================

for csv_path in csv_files:

    print("\n" + "=" * 60)
    print("Processing:", os.path.basename(csv_path))

    try:

        df = pd.read_csv(csv_path)

        original_rows = len(df)

        # ----------------------------------
        # Remove unnamed columns
        # ----------------------------------

        df = df.loc[
            :,
            ~df.columns.str.contains("^Unnamed")
        ]

        # ----------------------------------
        # Detect columns
        # ----------------------------------

        time_col = None
        rain_col = None

        for col in df.columns:

            if "time" in col.lower():
                time_col = col

            if "rain" in col.lower():
                rain_col = col

        if time_col is None:
            print("No timestamp column found")
            continue

        if rain_col is None:
            print("No rainfall column found")
            continue

        # ----------------------------------
        # Standardize names
        # ----------------------------------

        df = df.rename(
            columns={
                time_col: "timestamp",
                rain_col: "rainfall_mm"
            }
        )

        # ----------------------------------
        # Timestamp conversion
        # ----------------------------------

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            format="mixed",
            errors="coerce"
        )

        # ----------------------------------
        # Rainfall conversion
        # ----------------------------------

        df["rainfall_mm"] = (
            df["rainfall_mm"]
            .apply(convert_to_mm)
        )

        # ----------------------------------
        # Remove invalid rows
        # ----------------------------------

        nan_ts = (
            df["timestamp"]
            .isna()
            .sum()
        )

        nan_rain = (
            df["rainfall_mm"]
            .isna()
            .sum()
        )

        df = df.dropna(
            subset=[
                "timestamp",
                "rainfall_mm"
            ]
        )

        # ----------------------------------
        # Sort
        # ----------------------------------

        df = df.sort_values(
            "timestamp"
        ).reset_index(
            drop=True
        )

        # ----------------------------------
        # Save
        # ----------------------------------

        file_name = (
            os.path.basename(csv_path)
            .replace(".csv", "")
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            f"{file_name}_clean.csv"
        )

        df.to_csv(
            output_path,
            index=False
        )

        # ----------------------------------
        # Summary
        # ----------------------------------

        print(
            f"Original Rows: {original_rows}"
        )

        print(
            f"Final Rows: {len(df)}"
        )

        print(
            f"NaN timestamps removed: {nan_ts}"
        )

        print(
            f"NaN rainfall removed: {nan_rain}"
        )

        print(
            f"Saved: {output_path}"
        )

        print(
            "\nRainfall Distribution:"
        )

        print(
            df["rainfall_mm"]
            .value_counts()
            .sort_index()
            .head(20)
        )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )

print("\nCleaning Complete")

In [ ]:
#create wave timestamp dataframe

import os
import pandas as pd
from datetime import datetime

audio_root = "/kaggle/input/datasets/amarnathdj/oct-2025-rainfall-data"

audio_records = []

for root, dirs, files in os.walk(audio_root):

    for f in files:

        if not f.endswith(".wav"):
            continue

        try:
            ts = datetime.strptime(
                f.replace(".wav", ""),
                "%Y_%m_%d_%H_%M_%S_%f"
            )

            audio_records.append({
                "audio_path": os.path.join(root, f),
                "audio_time": ts
            })

        except:
            pass

audio_df = pd.DataFrame(audio_records)

print(audio_df.head())
print("Total wavs:", len(audio_df))

In [ ]:
#load csv
labels_df = pd.read_csv(
    "/kaggle/working/cleaned_labels_october_2025/October_2025_combined_labels.csv"
)

labels_df["timestamp"] = pd.to_datetime(
    labels_df["timestamp"],
    format="mixed"
)

print(labels_df.head())

# Optional sanity check
print(labels_df["timestamp"].min())
print(labels_df["timestamp"].max())


In [ ]:
import pandas as pd
import glob
import os

CSV_FOLDER = "/kaggle/working/cleaned_labels_october_2025"

csv_files = sorted(
    glob.glob(
        os.path.join(CSV_FOLDER, "*.csv")
    )
)

print("CSV files found:", len(csv_files))

dfs = []

for csv_file in csv_files:

    df = pd.read_csv(csv_file)

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed"
    )

    dfs.append(df)

labels_df = pd.concat(
    dfs,
    ignore_index=True
)

labels_df = labels_df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Total labels:", len(labels_df))
print(labels_df.head())

print(
    labels_df["timestamp"].min(),
    "->",
    labels_df["timestamp"].max()
)

In [ ]:
import os
from datetime import datetime

audio_root = "/kaggle/input/datasets/amarnathdj/oct-2025-rainfall-data"

audio_records = []

for root, dirs, files in os.walk(audio_root):

    for file in files:

        if not file.endswith(".wav"):
            continue

        try:

            ts = datetime.strptime(
                file.replace(".wav", ""),
                "%Y_%m_%d_%H_%M_%S_%f"
            )

            audio_records.append({
                "audio_time": ts,
                "audio_path": os.path.join(root, file)
            })

        except:
            pass

audio_df = pd.DataFrame(audio_records)

audio_df = audio_df.sort_values(
    "audio_time"
).reset_index(drop=True)

print("Total WAV files:", len(audio_df))

In [ ]:
#find wavs belonging to each label
from datetime import timedelta

aligned_samples = []

for _, row in labels_df.iterrows():

    label_time = row["timestamp"]

    start = label_time - timedelta(seconds=90)
    end   = label_time + timedelta(seconds=90)

    matching_wavs = audio_df[
        (audio_df["audio_time"] >= start) &
        (audio_df["audio_time"] <= end)
    ]

    aligned_samples.append({
        "timestamp": label_time,
        "rainfall_mm": row["rainfall_mm"],
        "wav_count": len(matching_wavs),
        "wav_files": matching_wavs["audio_path"].tolist()
    })

aligned_df = pd.DataFrame(aligned_samples)

aligned_df.head()

In [ ]:
#validate

print(
    aligned_df["wav_count"]
    .describe()
)

print(
    aligned_df["wav_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
#inspect wav spacing

audio_df = audio_df.sort_values("audio_time")

print(
    audio_df["audio_time"]
    .diff()
    .value_counts()
    .head(20)
)

In [ ]:
#remove invalid samples

aligned_df = aligned_df[
    aligned_df["wav_count"] >= 17
]

aligned_df = aligned_df.dropna(
    subset=["rainfall_mm"]
)

aligned_df = aligned_df.reset_index(drop=True)
month_name = "october_2025"
aligned_df.to_pickle(f"{month_name}_aligned_dataset.pkl")

In [ ]:
import pandas as pd
import glob
import os

csv_folder = "/kaggle/working/cleaned_labels_october_2025"

csv_files = sorted(
    glob.glob(
        os.path.join(csv_folder, "*.csv")
    )
)

print("CSV files found:", len(csv_files))

dfs = []

for csv_file in csv_files:

    df = pd.read_csv(csv_file)

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed"
    )

    dfs.append(df)

combined_df = pd.concat(
    dfs,
    ignore_index=True
)

combined_df = combined_df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Total rows:", len(combined_df))
print(combined_df.head())

combined_df.to_csv(
    "/kaggle/working/November_2024_combined_labels.csv",
    index=False
)

print("Saved: /kaggle/working/October_2025_combined_labels.csv")

In [ ]:
import os
import glob

folder = "/kaggle/working/cleaned_labels_october_2025"

# Delete all CSV files in the folder
for file in glob.glob(os.path.join(folder, "*.csv")):
    os.remove(file)

print("Old CSV files deleted")

# Save combined CSV into the same folder
combined_path = os.path.join(
    folder,
    "October_2025_combined_labels.csv"
)

combined_df.to_csv(
    combined_path,
    index=False
)

print("Saved:", combined_path)

# Verify contents
print("\nFolder contents:")
print(os.listdir(folder))

In [ ]:
import os

os.rename("december_2023_aligned_dataset.pkl", "april_2024_aligned_dataset.pkl")

In [ ]:
os.remove("/kaggle/working/f{month_name}_aligned_dataset.pkl")

In [ ]:
import pandas as pd

df = pd.read_pickle(
    "/kaggle/working/july_2024_aligned_dataset.pkl"
)

print(df.head())
print(df["timestamp"].min())
print(df["timestamp"].max())

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/cleaned_labels_november_2023/November_2023_clean.csv")

zero_count = (df["rainfall_mm"] == 0).sum()

print("Zero values:", zero_count)

In [ ]:
#!/usr/bin/env python3
"""
Align Mechanical Rainfall Dataset with Audio Files by Timestamp

This script:
1. Reads davis_label.csv (mechanical rain gauge timestamps and rainfall in mm)
2. Scans for corresponding audio WAV files within a time window
3. Creates aligned dataset pairs with verified timestamps
4. Generates alignment report with statistics
5. Outputs aligned_rainfall_audio_pairs.csv with verified matches

Author: Acoustic Rain Gauge Project
"""

import os
import sys
import csv
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Optional
import logging
from dataclasses import dataclass, asdict
import argparse


@dataclass
class RainfallLabel:
    """Represents a mechanical rain gauge reading"""
    timestamp: datetime
    rainfall_mm: float
    timestamp_str: str
    source: str = "davis_label.csv"


@dataclass
class AudioFile:
    """Represents an audio WAV file"""
    file_path: str
    file_name: str
    timestamp: datetime
    timestamp_str: str
    duration_sec: int = 10


@dataclass
class AlignedPair:
    """Represents an aligned rainfall label with corresponding audio file(s)"""
    label_timestamp: str
    rainfall_mm: float
    audio_file_path: str
    audio_file_name: str
    audio_timestamp: str
    time_diff_sec: float
    window_size_sec: int
    alignment_quality: str  # 'exact', 'good', 'acceptable'


class RainfallAudioAligner:
    """
    Aligns mechanical rainfall measurements with their corresponding audio files
    based on timestamp proximity.
    """

    def __init__(self, 
                 window_size_sec: int = 180,
                 exact_match_sec: int = 5,
                 logger: Optional[logging.Logger] = None):
        """
        Initialize the aligner.
        
        Args:
            window_size_sec: Time window (seconds) around each label to find audio files
                           Default 180 sec = 3 min (one inference cycle)
            exact_match_sec: Maximum time difference for 'exact' quality alignment
            logger: Python logger instance
        """
        self.window_size_sec = window_size_sec
        self.exact_match_sec = exact_match_sec
        self.logger = logger or self._setup_logger()
        
        self.rainfall_labels: List[RainfallLabel] = []
        self.audio_files: List[AudioFile] = []
        self.aligned_pairs: List[AlignedPair] = []
        self.alignment_report = {}

    @staticmethod
    def _setup_logger() -> logging.Logger:
        """Setup logging configuration"""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        return logging.getLogger(__name__)

    def parse_timestamp_from_filename(self, filename: str) -> Optional[datetime]:
        """
        Parse timestamp from audio filename.
        
        Supports formats:
        - YYYYMMDD_HHMMSS_*.wav (standard format)
        - YYYY-MM-DD_HH-MM-SS_*.wav
        - audio_YYYYMMDD_HHMMSS.wav
        
        Args:
            filename: Audio filename to parse
            
        Returns:
            Parsed datetime object or None if parsing fails
        """
        # Remove .wav extension
        name = filename.replace('.wav', '').replace('.WAV', '')
        
        # Try format: YYYYMMDD_HHMMSS
        patterns = [
            (r'(\d{8})_(\d{6})', '%Y%m%d_%H%M%S'),  # 20231201_101530
            (r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', '%Y-%m-%d_%H-%M-%S'),
            (r'audio_(\d{8})_(\d{6})', '%Y%m%d_%H%M%S'),  # audio_20231201_101530
        ]
        
        import re
        for pattern, date_fmt in patterns:
            match = re.search(pattern, name)
            if match:
                try:
                    # Reconstruct the timestamp string from match
                    if pattern == patterns[0][0]:  # First pattern
                        ts_str = f"{match.group(1)}_{match.group(2)}"
                    elif pattern == patterns[1][0]:  # Hyphenated format
                        ts_str = f"{match.group(1)}-{match.group(2)}-{match.group(3)}_{match.group(4)}-{match.group(5)}-{match.group(6)}"
                    else:  # audio_ prefix format
                        ts_str = f"{match.group(1)}_{match.group(2)}"
                    
                    return datetime.strptime(ts_str, date_fmt)
                except ValueError:
                    continue
        
        return None

    def parse_csv_timestamp(self, timestamp_str: str) -> Optional[datetime]:
        """
        Parse timestamp from CSV format.
        
        Supports:
        - ISO 8601: 2023-12-01T10:15:30
        - Standard: 2023-12-01 10:15:30
        - Unix-style: 1701416130
        
        Args:
            timestamp_str: Timestamp string from CSV
            
        Returns:
            Parsed datetime object or None if parsing fails
        """
        formats = [
            '%Y-%m-%dT%H:%M:%S',           # ISO 8601
            '%Y-%m-%d %H:%M:%S',           # Standard
            '%Y-%m-%d %H:%M:%S.%f',        # With microseconds
            '%d-%m-%Y %H:%M:%S',           # DD-MM-YYYY format
        ]
        
        # Try as Unix timestamp
        try:
            ts = float(timestamp_str)
            return datetime.fromtimestamp(ts)
        except (ValueError, OSError):
            pass
        
        # Try standard formats
        for fmt in formats:
            try:
                return datetime.strptime(timestamp_str.strip(), fmt)
            except ValueError:
                continue
        
        self.logger.warning(f"Could not parse timestamp: {timestamp_str}")
        return None

    def load_rainfall_csv(self, csv_path: str) -> bool:
        """
        Load mechanical rainfall measurements from CSV.
        
        Expected CSV format:
        timestamp,rainfall_mm
        2023-12-01 10:15:30,0.4
        2023-12-01 10:18:30,0.0
        
        Args:
            csv_path: Path to CSV file
            
        Returns:
            True if loaded successfully, False otherwise
        """
        if not os.path.exists(csv_path):
            self.logger.error(f"CSV file not found: {csv_path}")
            return False
        
        try:
            with open(csv_path, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    # Detect column names (case-insensitive)
                    ts_key = next((k for k in row.keys() if 'time' in k.lower()), None)
                    rain_key = next((k for k in row.keys() if 'rain' in k.lower()), None)
                    
                    if not ts_key or not rain_key:
                        self.logger.error("CSV must contain 'timestamp' and 'rainfall' columns")
                        return False
                    
                    ts_str = row[ts_key].strip()
                    try:
                        rainfall = float(row[rain_key])
                    except ValueError:
                        self.logger.warning(f"Invalid rainfall value: {row[rain_key]}, skipping")
                        continue
                    
                    dt = self.parse_csv_timestamp(ts_str)
                    if dt:
                        label = RainfallLabel(
                            timestamp=dt,
                            rainfall_mm=rainfall,
                            timestamp_str=ts_str
                        )
                        self.rainfall_labels.append(label)
            
            self.logger.info(f"Loaded {len(self.rainfall_labels)} rainfall labels from {csv_path}")
            return True
        
        except Exception as e:
            self.logger.error(f"Error loading CSV: {e}")
            return False

    def scan_audio_directory(self, audio_dir: str) -> bool:
        """
        Scan directory for audio WAV files and extract timestamps.
        
        Args:
            audio_dir: Path to directory containing audio files
            
        Returns:
            True if files found, False otherwise
        """
        if not os.path.isdir(audio_dir):
            self.logger.error(f"Audio directory not found: {audio_dir}")
            return False
        
        wav_count = 0
        parse_errors = 0
        
        for root, dirs, files in os.walk(audio_dir):
            for filename in sorted(files):
                if filename.lower().endswith('.wav'):
                    dt = self.parse_timestamp_from_filename(filename)
                    if dt:
                        full_path = os.path.join(root, filename)
                        audio = AudioFile(
                            file_path=full_path,
                            file_name=filename,
                            timestamp=dt,
                            timestamp_str=dt.isoformat()
                        )
                        self.audio_files.append(audio)
                        wav_count += 1
                    else:
                        parse_errors += 1
        
        self.logger.info(f"Found {wav_count} audio files with valid timestamps")
        if parse_errors > 0:
            self.logger.warning(f"Skipped {parse_errors} audio files with unparseable timestamps")
        
        return wav_count > 0

    def align_pairs(self) -> int:
        """
        Match rainfall labels with audio files based on timestamp proximity.
        
        For each rainfall label, finds all audio files within the window.
        Assigns quality scores: 'exact' (within 5s), 'good' (within 90s),
        'acceptable' (within window).
        
        Returns:
            Number of aligned pairs created
        """
        self.aligned_pairs = []
        
        for label in self.rainfall_labels:
            label_time = label.timestamp
            window_start = label_time - timedelta(seconds=self.window_size_sec//2)
            window_end = label_time + timedelta(seconds=self.window_size_sec//2)
            
            # Find all audio files in the window
            matching_audios = [
                audio for audio in self.audio_files
                if window_start <= audio.timestamp <= window_end
            ]
            
            if not matching_audios:
                continue
            
            # Create pair for each matching audio file
            for audio in matching_audios:
                time_diff = abs((audio.timestamp - label_time).total_seconds())
                
                # Assign quality based on time difference
                if time_diff <= self.exact_match_sec:
                    quality = 'exact'
                elif time_diff <= 90:
                    quality = 'good'
                else:
                    quality = 'acceptable'
                
                pair = AlignedPair(
                    label_timestamp=label.timestamp_str,
                    rainfall_mm=label.rainfall_mm,
                    audio_file_path=audio.file_path,
                    audio_file_name=audio.file_name,
                    audio_timestamp=audio.timestamp_str,
                    time_diff_sec=time_diff,
                    window_size_sec=self.window_size_sec,
                    alignment_quality=quality
                )
                self.aligned_pairs.append(pair)
        
        self.logger.info(f"Created {len(self.aligned_pairs)} aligned pairs")
        return len(self.aligned_pairs)

    def generate_alignment_report(self) -> Dict:
        """
        Generate statistics and quality report.
        
        Returns:
            Dictionary containing alignment statistics
        """
        if not self.aligned_pairs:
            self.logger.warning("No aligned pairs to report")
            return {}
        
        exact_count = sum(1 for p in self.aligned_pairs if p.alignment_quality == 'exact')
        good_count = sum(1 for p in self.aligned_pairs if p.alignment_quality == 'good')
        acceptable_count = sum(1 for p in self.aligned_pairs if p.alignment_quality == 'acceptable')
        
        time_diffs = [p.time_diff_sec for p in self.aligned_pairs]
        avg_time_diff = sum(time_diffs) / len(time_diffs) if time_diffs else 0
        max_time_diff = max(time_diffs) if time_diffs else 0
        min_time_diff = min(time_diffs) if time_diffs else 0
        
        self.alignment_report = {
            'total_rainfall_labels': len(self.rainfall_labels),
            'total_audio_files': len(self.audio_files),
            'total_aligned_pairs': len(self.aligned_pairs),
            'coverage_percent': (len(self.aligned_pairs) / len(self.rainfall_labels) * 100) if self.rainfall_labels else 0,
            'quality_breakdown': {
                'exact': exact_count,
                'good': good_count,
                'acceptable': acceptable_count
            },
            'time_difference_stats': {
                'average_sec': round(avg_time_diff, 2),
                'min_sec': round(min_time_diff, 2),
                'max_sec': round(max_time_diff, 2)
            },
            'window_size_sec': self.window_size_sec,
            'alignment_timestamp': datetime.now().isoformat()
        }
        
        return self.alignment_report

    def save_aligned_pairs_csv(self, output_path: str) -> bool:
        """
        Save aligned pairs to CSV file.
        
        Args:
            output_path: Output CSV file path
            
        Returns:
            True if saved successfully
        """
        try:
            with open(output_path, 'w', newline='') as f:
                fieldnames = [
                    'label_timestamp', 'rainfall_mm', 'audio_file_name',
                    'audio_file_path', 'audio_timestamp', 'time_diff_sec',
                    'alignment_quality', 'window_size_sec'
                ]
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                
                for pair in self.aligned_pairs:
                    row = {k: getattr(pair, k) for k in fieldnames}
                    writer.writerow(row)
            
            self.logger.info(f"Saved {len(self.aligned_pairs)} aligned pairs to {output_path}")
            return True
        
        except Exception as e:
            self.logger.error(f"Error saving CSV: {e}")
            return False

    def save_alignment_report_json(self, output_path: str) -> bool:
        """
        Save alignment report to JSON file.
        
        Args:
            output_path: Output JSON file path
            
        Returns:
            True if saved successfully
        """
        try:
            with open(output_path, 'w') as f:
                json.dump(self.alignment_report, f, indent=2)
            
            self.logger.info(f"Saved alignment report to {output_path}")
            return True
        
        except Exception as e:
            self.logger.error(f"Error saving report: {e}")
            return False

    def print_summary(self):
        """Print alignment summary to console"""
        if not self.alignment_report:
            print("No alignment data to display")
            return
        
        print("\n" + "="*70)
        print("RAINFALL-AUDIO ALIGNMENT SUMMARY")
        print("="*70)
        print(f"Timestamp: {self.alignment_report['alignment_timestamp']}")
        print(f"Window Size: {self.alignment_report['window_size_sec']} seconds")
        print()
        print(f"Rainfall Labels Loaded: {self.alignment_report['total_rainfall_labels']}")
        print(f"Audio Files Found: {self.alignment_report['total_audio_files']}")
        print(f"Aligned Pairs Created: {self.alignment_report['total_aligned_pairs']}")
        print(f"Coverage: {self.alignment_report['coverage_percent']:.1f}%")
        print()
        print("Quality Breakdown:")
        qual = self.alignment_report['quality_breakdown']
        print(f"  ✓ Exact matches (≤5s): {qual['exact']}")
        print(f"  ✓ Good matches (≤90s): {qual['good']}")
        print(f"  ✓ Acceptable matches (≤180s): {qual['acceptable']}")
        print()
        stats = self.alignment_report['time_difference_stats']
        print("Time Difference Statistics:")
        print(f"  Average: {stats['average_sec']:.2f} seconds")
        print(f"  Min: {stats['min_sec']:.2f} seconds")
        print(f"  Max: {stats['max_sec']:.2f} seconds")
        print("="*70 + "\n")


def main():
    """Main entry point"""
    parser = argparse.ArgumentParser(
        description='Align mechanical rainfall data with audio files by timestamp'
    )
    parser.add_argument(
        '--csv',
        required=True,
        help='Path to rainfall CSV file (davis_label.csv)'
    )
    parser.add_argument(
        '--audio-dir',
        required=True,
        help='Path to directory containing audio WAV files'
    )
    parser.add_argument(
        '--window',
        type=int,
        default=180,
        help='Time window in seconds to find matching audio (default: 180)'
    )
    parser.add_argument(
        '--output-csv',
        default='aligned_rainfall_audio_pairs.csv',
        help='Output CSV file for aligned pairs'
    )
    parser.add_argument(
        '--output-report',
        default='alignment_report.json',
        help='Output JSON file for alignment statistics'
    )
    parser.add_argument(
        '--verbose',
        action='store_true',
        help='Enable verbose logging'
    )
    
    args = parser.parse_args()
    
    # Setup logger
    log_level = logging.DEBUG if args.verbose else logging.INFO
    logging.basicConfig(
        level=log_level,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    logger = logging.getLogger(__name__)
    
    # Initialize aligner
    aligner = RainfallAudioAligner(window_size_sec=args.window, logger=logger)
    
    # Load data
    logger.info("Loading rainfall data...")
    if not aligner.load_rainfall_csv(args.csv):
        sys.exit(1)
    
    logger.info("Scanning audio directory...")
    if not aligner.scan_audio_directory(args.audio_dir):
        sys.exit(1)
    
    # Perform alignment
    logger.info("Aligning rainfall with audio files...")
    aligner.align_pairs()
    
    # Generate report
    aligner.generate_alignment_report()
    
    # Save outputs
    aligner.save_aligned_pairs_csv(args.output_csv)
    aligner.save_alignment_report_json(args.output_report)
    
    # Print summary
    aligner.print_summary()
    
    logger.info("Alignment complete!")


if __name__ == '__main__':
    main()
